# InferLite on free Google Colab GPUs (T4)

Measured timings only. Missing methods are labeled **unsupported**, never scored.

**If Colab says the session crashed / disk is full:** Runtime → Disconnect and delete runtime. Then reconnect with a T4 GPU. Do **not** `pip install -r requirements.txt` here — that reinstalls Torch and ONNX and is what fills the disk.

**Setup**

1. Runtime → Change runtime type → **T4 GPU**.
2. Run cells in order. Cell 1 clones `https://github.com/Shivani767/llm-inferlite` if needed.
3. Cell 2 installs a **small** Colab package list only. Colab already has PyTorch.

In [ ]:
import os, sys, pathlib, subprocess

def is_repo(path):
    return (pathlib.Path(path) / "backend" / "research").is_dir()

REPO_ROOT = None
for cand in [
    pathlib.Path("/content/llm-inferlite"),
    pathlib.Path("/content/llm-inferlite-main"),
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
    pathlib.Path("/content"),
]:
    if is_repo(cand):
        REPO_ROOT = cand
        break
    for name in ("llm-inferlite", "llm-inferlite-main"):
        nested = cand / name
        if is_repo(nested):
            REPO_ROOT = nested
            break
    if REPO_ROOT is not None:
        break

if REPO_ROOT is None:
    dest = pathlib.Path("/content/llm-inferlite")
    print("Repo not found. Cloning https://github.com/Shivani767/llm-inferlite ...")
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/Shivani767/llm-inferlite.git",
        str(dest),
    ])
    REPO_ROOT = dest
elif (REPO_ROOT / ".git").is_dir():
    # Old clones still have the disk-filling pip cell files; fast-forward to origin/main.
    subprocess.call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", "main"])
    subprocess.call(["git", "-C", str(REPO_ROOT), "reset", "--hard", "origin/main"])

BACKEND = REPO_ROOT / "backend"
assert (BACKEND / "research").exists(), (
    f"Could not find InferLite backend under {REPO_ROOT}. "
    "Run: !git clone --depth 1 https://github.com/Shivani767/llm-inferlite.git /content/llm-inferlite"
)
os.chdir(BACKEND)
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print("REPO_ROOT", REPO_ROOT)
print("BACKEND ", BACKEND)

In [ ]:
import shutil
from pathlib import Path

# Free leftover clones / pip cache from a crashed session. Do not reinstall torch.
for extra in (Path("/content/llm-inferlite-main"),):
    if extra.exists() and extra.resolve() != Path(REPO_ROOT).resolve():
        shutil.rmtree(extra, ignore_errors=True)

!pip cache purge -q || true
!df -h / | tail -1
!free -h | head -2

# Slim install only. NEVER: pip install -r requirements.txt on Colab.
!pip install -q -r requirements-colab.txt
print("install done")

In [ ]:
from research.capabilities import probe
from research.env import collect_environment
import json

env = collect_environment(seed=42)
caps = probe()
print("device:", caps["device"], "cuda:", caps["cuda"], "colab:", env.get("colab"))
print("gpu:", (env.get("torch") or {}).get("gpu"))
print("\nCapability matrix:")
for name, item in caps["experiments"].items():
    print(f"  [{'YES' if item['supported'] else 'NO ':3}] {name}: {item['reason']}")

## T4 lite suite

TinyLlama 1.1B. First pass times **fp16, INT8, INT4** when bitsandbytes works. AWQ/GPTQ/GGUF stay **unsupported** unless those libraries and files are present — they are not downloaded just to fill the disk.

After this succeeds you can run `configs/colab_t4.yaml` for KV / speculative / batching.

In [ ]:
from research.runner import load_config, run_config

cfg_path = REPO_ROOT / "configs" / "colab_t4_lite.yaml"
cfg = load_config(cfg_path)
cfg["results_dir"] = str(BACKEND / "results" / "colab_t4_lite")

summary = run_config(cfg, results_dir=cfg["results_dir"], make_plots=True)
print("measured", summary["n_measured"], "unsupported", summary["n_unsupported"], "error", summary["n_error"])
print("bundle", summary["bundle"])
print("csv", summary["csv"])
print("plots", summary["plots"])
print("pareto", json.dumps(summary["pareto"], indent=2, default=str)[:2000])

In [ ]:
from IPython.display import Image, display
from pathlib import Path

fig_dir = Path(cfg["results_dir"]) / "figures"
if fig_dir.exists():
    for p in sorted(fig_dir.glob("*.png")):
        print(p.name)
        display(Image(filename=str(p)))
else:
    print("No figures yet. Check n_measured in the previous cell.")

## Optional: download a GGUF and measure llama.cpp

Skip this cell if `llama-cpp-python` is not installed. InferLite will not fabricate GGUF numbers.

In [ ]:
from research.engine import run_benchmark

try:
    import llama_cpp  # noqa: F401
    rec = run_benchmark(
        model_id="TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF",
        method="gguf",
        backend="llama.cpp",
        gguf_file="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        filename="tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf",
        max_new_tokens=32,
        measure_runs=2,
        warmup_runs=1,
    )
    print(rec.status, rec.reason)
    print(rec.metrics.model_dump() if rec.metrics else None)
except Exception as exc:
    print("GGUF skipped:", type(exc).__name__, exc)

## Honesty checklist

- Do not copy old README tables that listed Llama-3-8B TensorRT-LLM TPS. Those were simulations.
- Cite only `status=measured` rows, with this notebook's environment dump.
- If a method is `unsupported`, report the reason, not a guessed speedup.